# C01–M3–L1 — Hands-on with GenAI: Choosing the Right Model for Your Application (Colab edition · Gemini stack)

Converted from the IBM Skills Network lab · [original notebook/files](https://github.com/MohammadWasiq0786/IBM-RAG-and-Agentic-AI/tree/main/01%20Develop%20Generative%20AI%20Applications/AI%20Assistant)

**Before you run:** add your free `GOOGLE_API_KEY` ([Google AI Studio](https://aistudio.google.com/apikey)) to Colab 🔑 Secrets and enable notebook access. No GPU needed — a CPU runtime is enough.

**What changed vs. the IBM version**
- The multi-file Flask project (`config.py`, `model.py`, `app.py`, `templates/index.html`, `llm_test.py`, `capital.py`) is recreated with `%%writefile` cells — same structure and file names.
- `ChatWatsonx` / `ModelInference` → `ChatGoogleGenerativeAI` built directly in each script (reads `GOOGLE_API_KEY` from the environment); the old `llm_factory.py` / `get_llm()` helper is gone.
- Model slots keep their names so `app.py` and `index.html` stay unchanged: **llama3** → `gemini-3.8-flash` · **granite** → `gemini-3.5-flash-lite` · **mixtral** → `gemini-2.5-flash`.
- watsonx `GenParams` / credentials in `config.py` → `temperature` / `max_output_tokens`; no project_id.
- Flask runs in a background thread and opens with `google.colab.output.serve_kernel_port_as_window(5000)`; optional Gradio wrapper with `share=True`.
- `llm_test.py` prints the parsed dicts (original `.content` on a dict would fail). Timing/quality comparison table and exercises at the end.

Everything else — explanations, code, exercises — is the original lab.

In [ ]:
# Minimal fast install (~5-10 seconds)
# + gradio for the optional front-end (flask, requests and pandas are preinstalled on Colab)
!pip install -q -U langchain-google-genai langchain-core langchain-chroma google-genai gradio

### 🔑 Gemini initialization
Run once. Creates `llm` (`gemini-3.8-flash`), `embeddings` (`models/gemini-embedding-2`) and the 1-line `llm_model()` alias used by older lab cells. Needs the free `GOOGLE_API_KEY` from [Google AI Studio](https://aistudio.google.com/apikey) in Colab 🔑 Secrets (notebook access on).

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Load API key securely from Colab Secrets (with graceful fallback)
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    pass

if not api_key:
    api_key = os.environ.get("GOOGLE_API_KEY")

if not api_key:
    import getpass
    print("🔑 Tip: Add 'GOOGLE_API_KEY' in the Colab 🔑 Secrets tab and toggle 'Notebook access' ON to load automatically.")
    api_key = getpass.getpass("Enter your Google AI Studio API key: ")

os.environ["GOOGLE_API_KEY"] = api_key

# Initialize LLM and Embeddings directly (no helper functions or custom shims)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.2)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# Backward-compatibility alias for older lab cells calling llm_model()
# (.text, not .content: Gemini 3.x can return content as a list of blocks; .text is always a str)
llm_model = lambda prompt, params=None: llm.invoke(prompt).text

print("✅ Google Gemini 3.8 Flash & Gemini Embedding 2 initialized successfully.")

## 1. Project files
The original lab is a small Flask app. Each `%%writefile` cell writes one file of the project into `/content`. The scripts create their own `ChatGoogleGenerativeAI` objects; `GOOGLE_API_KEY` (set in the init cell) is inherited by `!python …`.

### `capital.py` — first call to an LLM
The lab starts by calling a model directly with a Llama-3 formatted prompt (Gemini simply reads the special tokens as text; the plain prompt is in the comment).

In [ ]:
%%writefile capital.py
# Colab (Gemini): ibm_watsonx_ai ModelInference(ibm/granite-3-3-8b-instruct → later Llama 4 Maverick, greedy, 100 tokens) → gemini-3.8-flash
from langchain_google_genai import ChatGoogleGenerativeAI   # reads GOOGLE_API_KEY from the environment

model = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.0)

# text = """
# Only reply with the answer. What is the capital of Canada?
# """

text = """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert assistant who provides concise and accurate answers.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
What is the capital of Canada?<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
"""

print(model.invoke(text).text)

In [ ]:
!python capital.py

### 🧪 Lab exercise — "Now try doing it with other models"
In the IBM lab you rewrite the capital-city prompt with **Granite** (`<|system|> … <|user|> … <|assistant|>`) and **Mistral** (`<s>[INST] … [/INST]`) special tokens.

**Your task (Colab version):** ask *"What is the capital of France?"* with 4 prompt formats (plain, Llama, Granite, Mistral) against `gemini-3.8-flash`, `gemini-3.5-flash-lite` and `gemini-2.5-flash`, and compare the answers. Then send the same question as a `SystemMessage` + `HumanMessage` pair.

> Note: Gemini is a *chat* API, so special tokens are just extra text for it — expect the same answer in every format. They matter on raw text-completion endpoints such as watsonx `generate()`.

In [ ]:
# Your code here


> 💡 **✅ Answer cells** (code hidden in Colab) follow every exercise. Try the exercise in the cell above first. Then click **▸ Show code** on the ✅ cell to read the answer, or tick **run_answer** and run it. Unticked answer cells stop *Run all* on purpose, so you meet each exercise before its answer — use *Runtime → Run after* to continue.

In [ ]:
# @title ✅ Answer — Lab step "Now try doing it with other models" (special tokens)
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# Same question, 4 prompt formats × 3 models.
# On watsonx these formats matter (raw text completion). On Gemini (chat API) all four should give "Paris".
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

system = "You are an expert assistant who provides concise and accurate answers."
question = "What is the capital of France?"
prompts = {
    "plain":   f"Only reply with the answer. {question}",
    "llama":   (f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{system}<|eot_id|>\n"
                f"<|start_header_id|>user<|end_header_id|>\n{question}<|eot_id|>\n"
                f"<|start_header_id|>assistant<|end_header_id|>\n"),
    "granite": f"<|system|>\n{system}\n<|user|>\n{question}\n<|assistant|>\n",
    "mistral": f"<s>[INST] {system} [/INST]\n[INST] {question} [/INST]",   # no </s>: the model generates the end token
}
for model_id in ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-2.5-flash"]:
    m = ChatGoogleGenerativeAI(model=model_id, temperature=0.0)
    for name, p in prompts.items():
        print(f"{model_id:24} {name:8} → {m.invoke(p).text.strip()[:80]!r}")

# The chat-API equivalent of special tokens: role-tagged messages
print(m.invoke([SystemMessage(content=system), HumanMessage(content=question)]).text)

# ── What you should see ──
# Expected: every line says *Paris*. Gemini adds its own chat template, so the Llama, Granite and Mistral tokens are only extra text. The equivalent in a chat API is `SystemMessage` / `HumanMessage` (or `ChatPromptTemplate.from_messages`).

### `config.py` — model parameters and IDs

In [ ]:
%%writefile config.py
# Colab (Gemini): watsonx GenParams/credentials → ChatGoogleGenerativeAI settings. No project_id needed.

# Model parameters (greedy decoding on watsonx ≈ temperature 0)
PARAMETERS = {
    "temperature": 0.0,
    "max_output_tokens": 1024,   # was max_new_tokens=256; headroom for Gemini thinking tokens
}

# Model IDs — the three dropdown slots keep their original names (llama3 / granite / mixtral)
# so app.py and index.html stay unchanged. All three are free Gemini models.
LLAMA3_MODEL_ID = "gemini-3.8-flash"        # was meta-llama/llama-3-2-11b-vision-instruct
GRANITE_MODEL_ID = "gemini-3.5-flash-lite"  # was ibm/granite-3-8b-instruct (small, fast)
MIXTRAL_MODEL_ID = "gemini-2.5-flash"       # was mistralai/mistral-large (previous generation)

### `model.py` — one LangChain chain per model with `JsonOutputParser`
Each model keeps its original prompt format (Llama 3 tokens, Granite *System/Human/AI*, Mistral `[INST]`), then `prompt | model | json_parser` returns `{summary, sentiment, response}`.

In [ ]:
%%writefile model.py
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from config import PARAMETERS, LLAMA3_MODEL_ID, GRANITE_MODEL_ID, MIXTRAL_MODEL_ID
from langchain_google_genai import ChatGoogleGenerativeAI   # reads GOOGLE_API_KEY from the environment

# Define JSON output structure
class AIResponse(BaseModel):
    summary: str = Field(description="Summary of the user's message")
    sentiment: int = Field(description="Sentiment score from 0 (negative) to 100 (positive)")
    response: str = Field(description="Suggested response to the user")

# JSON output parser
json_parser = JsonOutputParser(pydantic_object=AIResponse)

# Function to initialize a model  (Colab: ChatWatsonx → ChatGoogleGenerativeAI)
def initialize_model(model_id):
    return ChatGoogleGenerativeAI(model=model_id, **PARAMETERS)

# Initialize models
llama3_llm = initialize_model(LLAMA3_MODEL_ID)
granite_llm = initialize_model(GRANITE_MODEL_ID)
mixtral_llm = initialize_model(MIXTRAL_MODEL_ID)

# Prompt templates — kept from the original lab to show model-specific prompt formats.
# (Gemini receives these strings as plain text in the user turn.)
llama3_template = PromptTemplate(
    template='''<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_prompt}\n{format_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>
{user_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
''',
    input_variables=["system_prompt", "format_prompt", "user_prompt"]
)

granite_template = PromptTemplate(
    template="System: {system_prompt}\n{format_prompt}\nHuman: {user_prompt}\nAI:",
    input_variables=["system_prompt", "format_prompt", "user_prompt"]
)

mixtral_template = PromptTemplate(
        template="<s>[INST]{system_prompt}\n{format_prompt}\n{user_prompt}[/INST]",
    input_variables=["system_prompt", "format_prompt", "user_prompt"]
)

def get_ai_response(model, template, system_prompt, user_prompt):
    chain = template | model | json_parser
    return chain.invoke({'system_prompt':system_prompt, 'user_prompt':user_prompt, 'format_prompt':json_parser.get_format_instructions()})

# Model-specific response functions
def llama3_response(system_prompt, user_prompt):
    return get_ai_response(llama3_llm, llama3_template, system_prompt, user_prompt)

def granite_response(system_prompt, user_prompt):
    return get_ai_response(granite_llm, granite_template, system_prompt, user_prompt)

def mixtral_response(system_prompt, user_prompt):
    return get_ai_response(mixtral_llm, mixtral_template, system_prompt, user_prompt)

### `llm_test.py` — call all three models from the command line

In [ ]:
%%writefile llm_test.py
from model import llama3_response, granite_response, mixtral_response

def call_all_models(system_prompt, user_prompt):
    llama_result = llama3_response(system_prompt, user_prompt)
    granite_result = granite_response(system_prompt, user_prompt)
    mixtral_result = mixtral_response(system_prompt, user_prompt)

    # JsonOutputParser returns dicts (the original printed .content, which fails on a dict)
    print("Llama3 Response:\n", llama_result)
    print("\nGranite Response:\n", granite_result)
    print("\nMixtral Response:\n", mixtral_result)

# Example call to test all models
call_all_models("You are a helpful assistant who provides concise and accurate answers", "What is the capital of Canada? Tell me a cool fact about it as well")

In [ ]:
!python llm_test.py

### `templates/index.html` — the UI (unchanged)

In [ ]:
!mkdir -p templates

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html>
<head>
    <title>AI Assistant</title>
</head>
<body>
    <h1>AI Assistant</h1>
    <form id="ai-form">
        <label for="message">Message:</label><br>
        <textarea id="message" name="message" rows="4" cols="50"></textarea><br><br>
        <label for="model">Model:</label><br>
        <select id="model" name="model">
            <option value="llama3">Llama3</option>
            <option value="granite">Granite</option>
            <option value="mixtral">Mixtral</option>
        </select><br><br>
        <input type="submit" value="Submit">
    </form>
    <br>
    <div id="response"></div>
    <script>
        document.getElementById('ai-form').addEventListener('submit', function(event) {
            event.preventDefault();
            var message = document.getElementById('message').value;
            var model = document.getElementById('model').value;
            
            fetch('/generate', {
                method: 'POST',
                headers: {
                    'Content-Type': 'application/json',
                },
                body: JSON.stringify({
                    'message': message,
                    'model': model
                }),
            })
            .then(response => response.json())
            .then(data => {
                if(data.error){
                    document.getElementById('response').innerText = 'Error: ' + data.error;
                } else {
                    document.getElementById('response').innerText = 'Response: ' + data.response + '\nDuration: ' + data.duration.toFixed(2) + ' seconds' + '\nFull JSON: ' + JSON.stringify(data);
                }
            })
            .catch((error) => {
                console.error('Error:', error);
                document.getElementById('response').innerText = 'Error: ' + error;
            });
        });
    </script>
</body>
</html>

### `app.py` — Flask routes (unchanged apart from the port)

In [ ]:
%%writefile app.py
from flask import Flask, request, jsonify, render_template
from model import llama3_response, granite_response, mixtral_response
import time

app = Flask(__name__)

@app.route('/', methods=['GET'])
def index():
    return render_template('index.html')

@app.route('/generate', methods=['POST'])
def generate():
    data = request.json
    user_message = data.get('message')
    model = data.get('model')
    
    if not user_message or not model:
        return jsonify({"error": "Missing message or model selection"}), 400
    
    system_prompt = "You are an AI assistant helping with customer inquiries. Provide a helpful and concise response."
    
    start_time = time.time()
    
    try:
        if model == 'llama3':
            result = llama3_response(system_prompt, user_message)
        elif model == 'granite':
            result = granite_response(system_prompt, user_message)
        elif model == 'mixtral':
            result = mixtral_response(system_prompt, user_message)
        else:
            return jsonify({"error": "Invalid model selection"}), 400
        
        result['duration'] = time.time() - start_time
        return jsonify(result)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == '__main__':
    app.run(port=5000, debug=True)

## 2. Run the app in Colab

In [ ]:
# Start the Flask app in a background thread (Colab swap for the Skills Network "Launch Application" port)
import threading, importlib, sys, time
sys.path.insert(0, "/content") if "/content" not in sys.path else None
import app as flask_app; importlib.reload(flask_app)
threading.Thread(target=lambda: flask_app.app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False), daemon=True).start()
time.sleep(3)

from google.colab import output
output.serve_kernel_port_as_window(5000)   # click the link to open the AI Assistant UI

In [ ]:
# Test the /generate endpoint without the browser
import requests, json
r = requests.post("http://127.0.0.1:5000/generate",
                  json={"message": "My order arrived late and the box was damaged.", "model": "llama3"})
print(json.dumps(r.json(), indent=2))

## 3. Choosing the right model — compare all three

In [ ]:
# Model comparison table — same prompt, all three models (record time + quality)
import time, pandas as pd
from model import llama3_response, granite_response, mixtral_response
system_prompt = "You are an AI assistant helping with customer inquiries. Provide a helpful and concise response."
msg = "I was charged twice for my subscription this month. Can you help?"
rows = []
for name, fn in [("llama3 → gemini-3.8-flash", llama3_response),
                 ("granite → gemini-3.5-flash-lite", granite_response),
                 ("mixtral → gemini-2.5-flash", mixtral_response)]:
    t = time.time()
    try:
        res = fn(system_prompt, msg); err = ""
    except Exception as e:
        res, err = {}, str(e)[:80]
    rows.append({"model": name, "seconds": round(time.time() - t, 2), "sentiment": res.get("sentiment"),
                 "summary": res.get("summary"), "response": (res.get("response") or err)[:120]})
pd.DataFrame(rows)

## 3b. 🧪 Lab exercise — Enhancing the JSON Structure
Add a new field to `AIResponse` that **recommends the next step the support representative should take** to resolve the issue.
1. Update the `AIResponse` class in `model.py` (re-run the `%%writefile model.py` cell after editing, or patch the file).
2. Modify the system prompt in `app.py` to mention this new field.
3. Test your changes with a variety of user messages.

> Keep the `response` field: `index.html` prints `data.response`. Re-running the Flask cell does **not** reload your code (port 5000 stays on the old server) — the answer below reloads the modules and serves on a new port, or use *Runtime → Restart session*.

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — Lab exercise "Enhancing the JSON Structure" (next-step field) (part 1/2)
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# 1) model.py — add a next_step field to AIResponse (keep `response`: the UI shows data.response)
import pathlib
p = pathlib.Path("model.py"); s = p.read_text()
old = '    response: str = Field(description="Suggested response to the user")\n'
new = old + '    next_step: str = Field(description="Recommended next step the support representative should take to resolve the issue")\n'
if "next_step" not in s:
    assert old in s, "AIResponse not found — re-run the model.py cell first"
    p.write_text(s.replace(old, new))

# 2) app.py — mention the new field in the system prompt
p = pathlib.Path("app.py"); s = p.read_text()
old = 'Provide a helpful and concise response."'
new = ('Provide a helpful and concise response, and recommend the next step the support '
       'representative should take to resolve the issue."')
if "next step" not in s:
    assert old in s
    p.write_text(s.replace(old, new))

# (UI) templates/index.html — show the next step under the response
p = pathlib.Path("templates/index.html"); s = p.read_text()
old = "'Response: ' + data.response +"
new = r"'Response: ' + data.response + '\nNext step: ' + data.next_step +"
if "Next step" not in s:
    assert old in s
    p.write_text(s.replace(old, new))
print("✅ files updated")

In [ ]:
# @title ✅ Answer — Lab exercise "Enhancing the JSON Structure" (next-step field) (part 2/2)
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# Helper: reload the edited files and serve them on a NEW port
# (the server already on 5000 keeps running the old code until you restart the session)
import importlib, threading, time, json
import config, model, app as flask_app
from google.colab import output

def relaunch(port):
    for mod in (config, model, flask_app):      # order matters: app imports model, model imports config
        importlib.reload(mod)
    threading.Thread(target=lambda: flask_app.app.run(host="0.0.0.0", port=port, debug=False,
                                                      use_reloader=False), daemon=True).start()
    time.sleep(2)
    output.serve_kernel_port_as_window(port)
    return flask_app.app.test_client()

client = relaunch(5001)

# 3) test with a variety of user messages
for msg in ["My order arrived late and the box was damaged.",
            "I can't log in after resetting my password.",
            "Do you ship to Canada, and how long does it take?",
            "Thanks, the replacement works perfectly!"]:
    r = client.post("/generate", json={"message": msg, "model": "llama3"})
    d = r.get_json()
    print(r.status_code, "|", msg, "\n   sentiment:", d.get("sentiment"), "\n   next_step:", d.get("next_step"), "\n")

# ── What you should see ──
# Expected: every reply now has a `next_step` key, for example *Arrange a replacement and send a prepaid return label* for the damaged-box message. Sentiment should be low for the complaints and high for the thank-you. IBM's own answer replaces `response` with `category` + `action`. That breaks the page, because `index.html` prints `data.response` (you'd see `Response: undefined`), so keep `response`.

## 4. (Optional) Gradio front-end

In [ ]:
# Optional: the same backend wrapped in Gradio (public share link)
import gradio as gr
from model import llama3_response, granite_response, mixtral_response
FNS = {"llama3": llama3_response, "granite": granite_response, "mixtral": mixtral_response}
def ask(message, model):
    return FNS[model]("You are an AI assistant helping with customer inquiries. Provide a helpful and concise response.", message)
gr.Interface(ask, [gr.Textbox(label="Message", lines=4), gr.Dropdown(list(FNS), value="llama3", label="Model")],
             gr.JSON(label="AI response"), title="AI Assistant").launch(share=True, debug=True)

## 🏋️ Colab practice exercises

1. Run the comparison cell with 3 different customer messages (angry, neutral, happy) and add a column scoring each answer 1–5 for quality. Which model would you ship, and why?
2. Replace the hand-written Llama/Granite/Mistral prompt strings with one `ChatPromptTemplate.from_messages([("system", ...), ("human", ...)])` and check whether JSON parsing failures go down.
3. Add a fourth model to `config.py` (e.g. `gemini-3.1-flash-lite` or `gemini-3.7-flash`), a `lite_response()` in `model.py`, a route branch in `app.py` and an `<option>` in `index.html`.
4. Add a `category` field (billing / shipping / technical / other) to `AIResponse` and show it in the UI.
5. Replace `JsonOutputParser` with `model.with_structured_output(AIResponse)` and compare reliability; make `/generate` return a friendly error instead of a 500 when parsing fails.

In [ ]:
# Your exercise code here


> 💡 The ✅ practice answers below build on each other (files + the `relaunch()` helper from the lab exercise in section 3b). Run them in order **after** the whole notebook; stop the Gradio cell (■) first.

In [ ]:
# @title ✅ Answer — 🏋️ Practice 1 — 3 tones × 3 models, quality column (part 1/2)
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

import time, pandas as pd
import model; importlib.reload(model)
from model import llama3_response, granite_response, mixtral_response

system_prompt = "You are an AI assistant helping with customer inquiries. Provide a helpful and concise response."
messages = {
    "angry":   "This is the THIRD time my package is late and nobody answers my emails. I want a refund NOW.",
    "neutral": "How do I change the email address on my account?",
    "happy":   "Just wanted to say your support team fixed my issue in minutes. Great service!",
}
MODELS = {"llama3 → gemini-3.8-flash": llama3_response,
          "granite → gemini-3.5-flash-lite": granite_response,
          "mixtral → gemini-2.5-flash": mixtral_response}
rows = []
for tone, msg in messages.items():
    for name, fn in MODELS.items():
        t = time.time()
        try:
            res, err = fn(system_prompt, msg), ""
        except Exception as e:
            res, err = {}, f"{type(e).__name__}: {str(e)[:60]}"
        rows.append({"tone": tone, "model": name, "seconds": round(time.time() - t, 2),
                     "sentiment": res.get("sentiment"), "response": (res.get("response") or err)[:150]})
df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 150)
df

In [ ]:
# @title ✅ Answer — 🏋️ Practice 1 — 3 tones × 3 models, quality column (part 2/2)
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# Read each response, then type YOUR 1–5 score per row (same order as df). Example scores:
df["quality (1-5)"] = [4, 4, 5,   4, 3, 4,   5, 4, 4]
summary = df.groupby("model")[["seconds", "quality (1-5)"]].mean().round(2)
summary["sentiment ok"] = df.groupby("model").apply(
    lambda g: bool((g.set_index("tone")["sentiment"].astype(float).diff().dropna() > 0).all()))  # angry < neutral < happy
summary

# ── What you should see ──
# How to decide: check the sentiment first (angry should be under 30, neutral around 50, happy over 70). Then look at the average quality and the time. The usual result: **ship `gemini-3.5-flash-lite` (the granite slot)** for a high-volume support assistant. It's the fastest and cheapest, and its short answers are good enough when the scores are within about 0.5. Choose `gemini-3.8-flash` when answer quality matters more than speed (it handles angry or complex messages better). `gemini-2.5-flash` is an older generation with a lower free quota, so you rarely need it. Use your own table to decide; the lab's point is to test on your own use case.

In [ ]:
# @title ✅ Answer — 🏋️ Practice 2 — one ChatPromptTemplate for all models + parse-failure count
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.exceptions import OutputParserException
from pydantic import ValidationError
from model import (json_parser, AIResponse, llama3_llm, granite_llm, mixtral_llm,
                   llama3_template, granite_template, mixtral_template)

chat_template = ChatPromptTemplate.from_messages([
    ("system", "{system_prompt}\n{format_prompt}"),
    ("human", "{user_prompt}"),
])
inputs = {"system_prompt": system_prompt, "format_prompt": json_parser.get_format_instructions()}

def failures(template, llm, msgs):
    bad = 0
    for m in msgs:
        try:
            AIResponse.model_validate((template | llm | json_parser).invoke({**inputs, "user_prompt": m}))
        except (OutputParserException, ValidationError):
            bad += 1
    return bad

test_msgs = list(messages.values()) * 2          # 6 calls per row → 36 calls total (mind the free-tier limit)
for name, llm, own in [("llama3", llama3_llm, llama3_template),
                       ("granite", granite_llm, granite_template),
                       ("mixtral", mixtral_llm, mixtral_template)]:
    print(f"{name:8} own template: {failures(own, llm, test_msgs)}/{len(test_msgs)} failures | "
          f"ChatPromptTemplate: {failures(chat_template, llm, test_msgs)}/{len(test_msgs)} failures")

# ── What you should see ──
# Expected: with Gemini, both are usually at 0 failures, because Gemini follows the format instructions well either way. `ChatPromptTemplate` is still the better choice. It sends a real system message (the chat equivalent of special tokens), one template works for every chat model, and you can't get the tokens wrong. With raw-completion models (watsonx `generate`), mismatched special tokens are what cause the failures.

In [ ]:
# @title ✅ Answer — 🏋️ Practice 3 — add a fourth model end-to-end
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

import pathlib

def patch(path, old, new, marker):
    p = pathlib.Path(path); s = p.read_text()
    if marker in s: return
    assert old in s, f"pattern not found in {path}"
    p.write_text(s.replace(old, new, 1))

# config.py — new model ID
patch("config.py", 'MIXTRAL_MODEL_ID = "gemini-2.5-flash"',
      'MIXTRAL_MODEL_ID = "gemini-2.5-flash"\nLITE_MODEL_ID = "gemini-3.1-flash-lite"   # 4th model', "LITE_MODEL_ID")

# model.py — model + response function (reuses the Granite-style template)
pathlib.Path("model.py").write_text(pathlib.Path("model.py").read_text() + '''
# --- Practice 3: fourth model ---
from config import LITE_MODEL_ID
lite_llm = initialize_model(LITE_MODEL_ID)

def lite_response(system_prompt, user_prompt):
    return get_ai_response(lite_llm, granite_template, system_prompt, user_prompt)
''') if "lite_response" not in pathlib.Path("model.py").read_text() else None

# app.py — import + route branch
patch("app.py", "from model import llama3_response, granite_response, mixtral_response",
      "from model import llama3_response, granite_response, mixtral_response, lite_response", "lite_response")
patch("app.py", "        else:\n            return jsonify({\"error\": \"Invalid model selection\"}), 400",
      "        elif model == 'lite':\n            result = lite_response(system_prompt, user_message)\n"
      "        else:\n            return jsonify({\"error\": \"Invalid model selection\"}), 400", "== 'lite'")

# index.html — dropdown option
patch("templates/index.html", '<option value="mixtral">Mixtral</option>',
      '<option value="mixtral">Mixtral</option>\n            <option value="lite">Flash-Lite 3.1</option>', 'value="lite"')

client = relaunch(5002)
r = client.post("/generate", json={"message": "Where is my order #1234?", "model": "lite"})
print(r.status_code, r.get_json())

# ── What you should see ──
# Expected: `200` and a JSON reply from `gemini-3.1-flash-lite`, and *Flash-Lite 3.1* appears in the dropdown on port 5002. For Gradio, add `"lite": lite_response` to `FNS`.

In [ ]:
# @title ✅ Answer — 🏋️ Practice 4 — `category` field shown in the UI
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# model.py — category with a fixed set of values
patch("model.py", "from pydantic import BaseModel, Field",
      "from typing import Literal\nfrom pydantic import BaseModel, Field", "from typing import Literal")
patch("model.py",
      '    response: str = Field(description="Suggested response to the user")\n',
      '    response: str = Field(description="Suggested response to the user")\n'
      '    category: Literal["billing", "shipping", "technical", "other"] = Field(description="Category of the inquiry")\n',
      "category: Literal")

# index.html — show it
patch("templates/index.html", "'Response: ' + data.response +",
      r"'Category: ' + data.category + '\nResponse: ' + data.response +", "data.category")

client = relaunch(5003)
for msg, expected in [("I was charged twice this month.", "billing"),
                      ("My parcel is stuck in customs.", "shipping"),
                      ("The app crashes when I upload a photo.", "technical"),
                      ("Do you have a student discount program?", "other")]:
    d = client.post("/generate", json={"message": msg, "model": "llama3"}).get_json()
    print(f"{d.get('category')!s:10} (expected {expected}) ← {msg}")

# ── What you should see ──
# Expected: all four categories match. `Literal[...]` appears in the format instructions as an `enum`, so the model knows the allowed values. `JsonOutputParser` still doesn't enforce them (see Practice 5).

In [ ]:
# @title ✅ Answer — 🏋️ Practice 5 — `with_structured_output` + friendly error
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# model.py — structured-output versions of the response functions (appended; they override the old ones)
pathlib.Path("model.py").write_text(pathlib.Path("model.py").read_text() + '''
# --- Practice 5: structured output (schema-validated) instead of JsonOutputParser ---
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.exceptions import OutputParserException

structured_prompt = ChatPromptTemplate.from_messages([("system", "{system_prompt}"), ("human", "{user_prompt}")])

def get_structured_response(model, system_prompt, user_prompt):
    chain = structured_prompt | model.with_structured_output(AIResponse)
    result = chain.invoke({"system_prompt": system_prompt, "user_prompt": user_prompt})
    if result is None:                      # model returned nothing parseable
        raise OutputParserException("Empty structured output")
    return result.model_dump()

def llama3_response(system_prompt, user_prompt):
    return get_structured_response(llama3_llm, system_prompt, user_prompt)

def granite_response(system_prompt, user_prompt):
    return get_structured_response(granite_llm, system_prompt, user_prompt)

def mixtral_response(system_prompt, user_prompt):
    return get_structured_response(mixtral_llm, system_prompt, user_prompt)

if "lite_llm" in globals():
    def lite_response(system_prompt, user_prompt):
        return get_structured_response(lite_llm, system_prompt, user_prompt)
''') if "get_structured_response" not in pathlib.Path("model.py").read_text() else None

# app.py — friendly errors instead of a raw 500
patch("app.py", "import time\n",
      "import time\nfrom langchain_core.exceptions import OutputParserException\nfrom pydantic import ValidationError\n",
      "OutputParserException")
patch("app.py",
      '    except Exception as e:\n        return jsonify({"error": str(e)}), 500',
      '    except (OutputParserException, ValidationError):\n'
      '        return jsonify({"error": "Sorry, the AI answer could not be read. Please try again or pick another model."}), 502\n'
      '    except Exception as e:\n'
      '        return jsonify({"error": f"The AI service is unavailable right now ({type(e).__name__}). Please try again in a minute."}), 503',
      "could not be read")

client = relaunch(5004)
d = client.post("/generate", json={"message": "I was charged twice this month.", "model": "granite"}).get_json()
print(d)

# Reliability comparison: old JsonOutputParser chain vs structured output (same messages)
import model
def count_ok(fn, msgs):
    ok = 0
    for m in msgs:
        try:
            model.AIResponse.model_validate(fn(m)); ok += 1
        except Exception:
            pass
    return ok
msgs = list(messages.values())
old_fn = lambda m: model.get_ai_response(model.granite_llm, model.granite_template, system_prompt, m)
new_fn = lambda m: model.granite_response(system_prompt, m)
print("JsonOutputParser valid:", count_ok(old_fn, msgs), "/", len(msgs))
print("with_structured_output valid:", count_ok(new_fn, msgs), "/", len(msgs))

# Prove the friendly error: make the granite branch fail with a parsing error
real = flask_app.granite_response
flask_app.granite_response = lambda s, u: (_ for _ in ()).throw(OutputParserException("bad JSON"))
r = client.post("/generate", json={"message": "test", "model": "granite"})
print(r.status_code, r.get_json())            # → 502 {'error': 'Sorry, the AI answer could not be read…'}
flask_app.granite_response = real

# ── What you should see ──
# Expected: the structured version always returns a valid `AIResponse` (correct types, `category` limited to the 4 values), because Gemini is constrained by a JSON schema instead of reading format instructions in the prompt. The old chain usually passes too, but nothing guarantees it. A parsing failure now returns `502` with a readable message instead of a `500` stack-trace string.